# PyIRI Tutorial: Global Daily Ionospheric Parameters and Electron Density

This tutorial demonstrates how to use the **PyIRI** model (with its spherical harmonics architecture) to compute and visualize global maps of ionospheric parameters for a specified month, year, day, and solar activity F10.7 index.

Specifically, this example shows how to:

- Configure the PyIRI model for a given date
- Specify the **foF2** and **hmF2** models (e.g., CCIR or URSI for foF2; SHU2015, AMTB2013, or BSE1979 for hmF2)
- Generate a **horizontal global grid** in **geographic coordinates (GEO)**
- Evaluate ionospheric parameters at each grid point over a full 24-hour UT cycle

> **Note**: This example uses **GEO** input coordinates (longitude, latitude).
> The PyIRI model also supports input in:
> - **Magnetic Local Time & Quasi-Dipole Latitude (MLT)**
> - **Quasi-Dipole Longitude & Latitude (QD)**  
> See the coordinate transformation tutorial for how to generate and use these input formats.

---

### Output

The output consists of gridded global maps of key ionospheric parameters. For example, the `F2` dictionary includes:

- **foF2** – Peak plasma frequency of the F2 layer  
- **hmF2** – Peak height of the F2 layer  
- **B0** and **B1** – Thickness and shape parameters defining the electron density profile  

All output maps have the shape **(N_T, N_G)**, where:
- `N_T` is the number of time points (from the UT array)
- `N_G` is the number of horizontal grid locations

The EDP output has the shape **(N_T, N_V, N_G)**, where:
- `N_V` is the number of vertical points (from the altitude array)

In [1]:
# Import libraries
import numpy as np
import PyIRI
import PyIRI.sh_library as sh  # Updated PyIRI using spherical harmonics
import matplotlib.pyplot as plt

<frozen importlib._bootstrap>:491: RuntimeWarning: The global interpreter lock (GIL) has been enabled to load module 'netCDF4._netCDF4', which has not declared that it can run safely without the GIL. To override this behavior and keep the GIL disabled (at your own risk), run with PYTHON_GIL=0 or -Xgil=0.


In [2]:
import numcodecs
numcodecs.blosc.use_threads = False

import zarr
# compressor = zarr.codecs.BloscCodec(
#         cname="zstd",
#         clevel=22,
#         shuffle=zarr.codecs.BloscShuffle.shuffle
#     )

compressor = zarr.codecs.ZstdCodec(level=22)

c:\winpython\python\Lib\site-packages\google_crc32c\__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


In [3]:
def zarr_dict2group(zg: zarr.Group, name: str, dict: dict, overwrite: bool = True):
    grp = zg.create_group(name, overwrite=overwrite)
    for key, val in dict.items():
        arr = grp.create_array(key, shape = val.shape, dtype = val.dtype, overwrite=overwrite, compressors=compressor)
        arr[:] = val

In [4]:
# Specify solar activity index (F10.7 in SFU)
# https://kp.gfz.de/en/data#c222
# https://lasp.colorado.edu/lisird/data/noaa_radio_flux
# https://omniweb.gsfc.nasa.gov/form/dx1.html
F107 = 100

# Create horizontal grid
lon_res = 5
lat_res = 5
alon_2d, alat_2d = np.mgrid[-180:180 + lon_res:lon_res, -90:90 + lat_res:lat_res]
alon = np.reshape(alon_2d, alon_2d.size)
alat = np.reshape(alat_2d, alat_2d.size)

# Time grid: Universal Time from 0 to 24 in 15-minute steps
hr_res = 1
aUT = np.arange(0, 24, hr_res)

# Height grid: 90 km to 700 km in 1 km steps
alt_res = 1
alt_min = 90
alt_max = 700
aalt = np.arange(alt_min, alt_max, alt_res)

# Coefficient sources and model options
foF2_coeff = 'CCIR'       # Options: 'CCIR' or 'URSI'
hmF2_model = 'SHU2015'    # Options: 'SHU2015', 'AMTB2013', 'BSE1979'
coord = 'GEO'             # Coordinate system: 'GEO', 'QD', or 'MLT'
coeff_dir = None          # Use default coefficient path


In [ ]:
# Specify date
year = 1997
month = 1
day = 1

# ----------------------------------------
# Run PyIRI (Spherical Harmonics version)
# ----------------------------------------
# For each day, compute ionospheric parameters for F2, F1, and E layers
# for day in range(18, 18 + 1):
F2, F1, E, sun, mag, EDP = sh.IRI_density_1day(year,
                                            month,
                                            day,
                                            aUT,
                                            alon,
                                            alat,
                                            aalt,
                                            F107,
                                            coeff_dir=coeff_dir,
                                            foF2_coeff=foF2_coeff,
                                            hmF2_model=hmF2_model,
                                            coord=coord)
# save parameters to zarr
save_as = f'daily_parameters/{year}_{month}_{day}.zarr'
z_root = zarr.create_group(save_as, overwrite=True)
zarr_dict2group(z_root, 'F2', F2)
zarr_dict2group(z_root, 'F1', F1)
zarr_dict2group(z_root, 'E', E)
zarr_dict2group(z_root, 'sun', sun)
zarr_dict2group(z_root, 'mag', mag)
EDP_save = z_root.create_array('EDP', shape = EDP.shape, dtype = EDP.dtype, overwrite=True, compressors=compressor)
EDP_save[:] = EDP
print(f'Saved daily data to {save_as}')

Saved daily data to daily_parameters/d_1997_1_1.zarr


In [6]:
EDP

array([[[5.89828389e+09, 6.68436474e+09, 7.38343929e+09, ...,
         8.96195831e+08, 6.86350103e+08, 5.70783461e+08],
        [7.14714420e+09, 8.09966417e+09, 8.94675576e+09, ...,
         1.08594991e+09, 8.31672949e+08, 6.91637057e+08],
        [8.64557053e+09, 9.79778998e+09, 1.08224776e+10, ...,
         1.31362349e+09, 1.00603639e+09, 8.36641430e+08],
        ...,
        [1.63614907e+10, 1.83822666e+10, 2.01909543e+10, ...,
         1.02433968e+10, 9.54034102e+09, 1.31128909e+10],
        [1.62896816e+10, 1.82980944e+10, 2.00975676e+10, ...,
         1.01990429e+10, 9.49754875e+09, 1.30522983e+10],
        [1.62183495e+10, 1.82145032e+10, 2.00048316e+10, ...,
         1.01549853e+10, 9.45504987e+09, 1.29921298e+10]],

       [[5.89993855e+09, 6.64727215e+09, 7.31306969e+09, ...,
         8.69435769e+08, 6.78563991e+08, 5.70612553e+08],
        [7.14914922e+09, 8.05471787e+09, 8.86148661e+09, ...,
         1.05352386e+09, 8.22238263e+08, 6.91429962e+08],
        [8.64799590e+09, 

Read daily parameters from .h5 file saved from the cell above

In [ ]:
# z_root = zarr.open_group(save_as, mode='r')


# for key in z_root.keys():
#     print(f'Group: {key}')
#     for subkey in z_root[key].keys():
#         print(f'  Dataset: {subkey}, shape: {z_root[key][subkey].shape}, dtype: {z_root[key][subkey].dtype}')

In [ ]:

filename = 'daily_parameters/2025_11_17.h5'

z_root = zarr.open_group(filename, mode='r')
# load each dictionary from group
F2 = dict((key, np.array(z_root['F2'][key])) for key in z_root['F2'].keys())
F1 = dict((key, np.array(z_root['F1'][key])) for key in z_root['F1'].keys())
E = dict((key, np.array(z_root['E'][key])) for key in z_root['E'].keys())
sun = dict((key, np.array(z_root['sun'][key])) for key in z_root['sun'].keys())
mag = dict((key, np.array(z_root['mag'][key])) for key in z_root['mag'].keys())

# load numpy array directly
EDP = np.array(z_root['EDP'])

### Plot results for parameters

In [ ]:
# Select a time frame to plot
UT_plot = 10

ind_time = np.where(aUT == UT_plot)

# Plot foF2
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F2['fo'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=2, vmax=14)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$fo$F2 (MHz)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_foF2.png', format='png', bbox_inches='tight')


# Plot hmF2
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F2['hm'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=200, vmax=360)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$hm$F2 (km)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_hmF2.png', format='png', bbox_inches='tight')


# Plot B0
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F2['B0'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=60, vmax=220)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$B0$ (km)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_B0.png', format='png', bbox_inches='tight')


# Plot B1
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F2['B1'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=5)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$B1$ (unitless)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_B1.png', format='png', bbox_inches='tight')


# Plot B_top
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F2['B_top'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=36, vmax=46)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$B_{top}$ (km)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_B_top.png', format='png', bbox_inches='tight')


### Plot results for F1 layer

In [ ]:
# Plot probability of F1 to occurre
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F1['P'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=1)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('Probability of F1')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_P.png', format='png', bbox_inches='tight')


# Plot thickness of F1
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F1['B_bot'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=50)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('B$_{bot}^{F1}$ (km)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_B_F1_bot.png', format='png', bbox_inches='tight')


# Plot foF1
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F1['fo'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=10)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$fo$F1 (MHz)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_foF1.png', format='png', bbox_inches='tight')


# Plot hmF1
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(F1['hm'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=100, vmax=200)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$hm$F1 (km)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_hmF1.png', format='png', bbox_inches='tight')

### Plot results for E region

In [ ]:
# Plot foE
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(E['fo'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=4)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$fo$E (MHz)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_foE.png', format='png', bbox_inches='tight')


### Calculate vertical TEC from EDP array

In [ ]:
TEC = PyIRI.main_library.edp_to_vtec(EDP, aalt, min_alt=0.0, max_alt=202000.0)

### Plot vTEC

In [ ]:
# Plot vTEC
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(TEC[ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=60)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('vTEC (TECU)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_vTEC.png', format='png', bbox_inches='tight')

### Plot electron density vertical profiles from one location

In [ ]:
# Select location to plot EDP
lon_plot = 10
lat_plot = 10


fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(4, 4),
                           constrained_layout=True)
ax.set_xlabel('Electron Density (m$^{-3}$)')
ax.set_ylabel('Altitude (km)')
ax.set_facecolor("lightgrey")
plt.xlim([0, 1.4e12])
plt.ylim([0, 700])
ind_grid = np.where((alon == lon_plot) & (alat == lat_plot))
ind_time = np.where(aUT == UT_plot)
ind_vert = np.where(aalt >= 0)
ind = ind_time, ind_vert, ind_grid
x = np.reshape(EDP[ind], aalt.shape)
ax.plot(x, aalt, c='black', linewidth=1)
plt.title(str(lon_plot) + '° Lon, ' + str(lat_plot) + '° Lat, ' + str(UT_plot) + ' UT')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_EDP.png', format='png', bbox_inches='tight')

### Compute the sporadic E layer critical frequency

Because the sporadic E layer contains sharper density gradients, a higher number of spherical harmonic coefficients is needed to reconstruct it accurately. The sporadic E layer parameter functions are therefore decoupled from the ionospheric parameter computation of the F2, F1, and E layers.

In [ ]:
# Compute ionospheric parameters for sporadic E layer
Es = sh.sporadic_E_1day(year,
                        month,
                        day,
                        aUT,
                        alon,
                        alat,
                        F107,
                        coeff_dir=coeff_dir,
                        coord=coord)

### Plot foEs

In [ ]:
# Plot foEs
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5, 3),
                        constrained_layout=True)
plt.xlim([-180, 180])
plt.ylim([-90, 90])
plt.xticks(np.arange(-180, 180 + 45, 90))
plt.yticks(np.arange(-90, 90 + 45, 45))
ax.set_facecolor('grey')
ax.set_xlabel('Geo Lon (°)')
ax.set_ylabel('Geo Lat (°)')
z = np.reshape(Es['fo'][ind_time, :], alon_2d.shape)
mesh = ax.pcolormesh(alon_2d, alat_2d, z, vmin=0, vmax=10)
ax.scatter(sun['lon'][ind_time], sun['lat'][ind_time],
                    c='red', s=20, edgecolors="black", linewidths=0.5)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('$fo$Es (MHz)')
# Save figure
plot_dir = 'figure/'
plt.savefig(plot_dir + 'PyIRI_sh_foEs.png', format='png', bbox_inches='tight')